# Mefloquine

# Packages

In [2]:
from cmapPy.pandasGEXpress import parse
import networkx as nx
import numpy as np
import pandas as pd
import pickle
import random

# Directories

In [3]:
INPUT = 'D:/DDesktop/_work/data/canada/input/'
OUTPUT = 'D:/DDesktop/_work/data/canada/output/'

GRAPH = 'D:/DDesktop/_work/graphs/canada/'

# Functions

In [4]:
def pickle_load(path: str, report: bool = False):
    '''
    Loads pickled data.
    '''

    with open(path, 'rb') as f:
        data = pickle.load(f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_load')
            print(f'Pickled graph loaded w/ {num_nodes:,} nodes and {num_edges:,} edges')
            print()
        else:
            print('>> pickle_load')
            print(f'Pickled file loaded')
            print()

    return data

def pickle_save(path: str, data, report: bool = False):
    '''
    Pickles data.
    '''

    with open(path, 'wb') as f:
        pickle.dump(data, f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_save')
            print(f'Graph w/ {num_nodes:,} nodes and {num_edges:,} edges pickled')
        else:
            print('>> pickle_save')
            print(f'Data pickled')
            print()

# Processing

## STRING

### `df_string_edgelist`

In [5]:
# Load data
df_string_edgelist = pd.read_csv(INPUT + '9606.protein.physical.links.full.v12.0.txt.gz', compression = 'gzip', sep = ' ')

# Drop columns
df_string_edgelist = df_string_edgelist[['protein1', 'protein2', 'combined_score']]

# Remove taxon ID
for column in ['protein1', 'protein2']:
    df_string_edgelist[column] = df_string_edgelist[column].str.replace('9606.', '')

# Rename columns
for old, new in zip(['protein1', 'protein2', 'combined_score'], ['source', 'target', 'weight']):
    df_string_edgelist.rename(columns = {old : new}, inplace = True)

# Save data
df_string_edgelist.to_csv(OUTPUT + 'df_string_edgelist.csv', index = False)
# Show data
df_string_edgelist.head()

,source,target,weight
0,ENSP00000000233,ENSP00000257770,311
1,ENSP00000000233,ENSP00000226004,161
2,ENSP00000000233,ENSP00000434442,499
3,ENSP00000000233,ENSP00000262455,531
4,ENSP00000000233,ENSP00000303145,499


### `df_string_info`

In [6]:
# Load data
df_string_info = pd.read_csv(INPUT + '9606.protein.info.v12.0.txt.gz', compression = 'gzip', sep = '\t')

# Drop columns
df_string_info.drop(columns = 'protein_size', inplace = True)

# Remove taxon ID
df_string_info['#string_protein_id'] = df_string_info['#string_protein_id'].str.replace('9606.', '')

# Rename columns
for old, new in zip(list(df_string_info.columns), ['id', 'string_name', 'string_desc']):
    df_string_info.rename(columns = {old : new}, inplace = True)

# Save data
df_string_info.to_csv(OUTPUT + 'df_string_info.csv', index = False)
# Show data
df_string_info.head()

,id,string_name,string_desc
0,ENSP00000000233,ARF5,ADP-ribosylation factor 5; GTP-binding protein...
1,ENSP00000000412,M6PR,Cation-dependent mannose-6-phosphate receptor;...
2,ENSP00000001008,FKBP4,"Peptidyl-prolyl cis-trans isomerase FKBP4, N-t..."
3,ENSP00000001146,CYP26B1,Cytochrome P450 26B1; Involved in the metaboli...
4,ENSP00000002125,NDUFAF7,"Protein arginine methyltransferase NDUFAF7, mi..."


### `df_string`

In [7]:
# # Load data
# df_string = pd.read_csv(OUTPUT + 'df_string_edgelist.csv')
# df_string_info = pd.read_csv(OUTPUT + 'df_string_info.csv')

# # Rename columns
# df_string_info.rename(columns = {'id' : 'source'}, inplace = True)
# # Merge
# df_string = pd.merge(df_string, df_string_info, how = 'left', on = 'source')
# # Rename columns
# df_string_info.rename(columns = {'source' : 'target'}, inplace = True)
# # Merge
# df_string = pd.merge(df_string, df_string_info, how = 'left', on = 'target')
# # Reset column name
# df_string_info.rename(columns = {'target' : 'id'}, inplace = True)

# # Rename columns
# for old, new in zip(list(df_string.columns), ['source', 'target', 'weight', 'source_string_name'])

# # Save data
# #df_string.to_csv(OUTPUT + 'df_string.csv', index = False)
# # Show data
# df_string.head()

## OPENtargets

### `df_opentargets`

In [8]:
# Load data
df_opentargets = pd.read_csv(INPUT + 'EFO_0007444-known-drugs.tsv', sep = '\t')

# Drop columns
df_opentargets.drop(columns = ['diseaseId', 'diseaseName'], inplace = True)
# Lower case
df_opentargets['drugName'] = df_opentargets['drugName'].str.lower()

# Save data
df_opentargets.to_csv(OUTPUT + 'df_opentargets.csv', index = False)
# Show data
df_opentargets.head()

,drugId,drugName,type,mechanismOfAction,actionType,symbol,name,phase,status,source
0,CHEMBL112,acetaminophen,Small molecule,Anandamide amidohydrolase inhibitor,Inhibitor,FAAH,fatty acid amide hydrolase,3.0,Completed,https://clinicaltrials.gov/study/NCT02974348
1,CHEMBL112,acetaminophen,Small molecule,Cyclooxygenase inhibitor,Inhibitor,PTGS2,prostaglandin-endoperoxide synthase 2,3.0,Completed,https://clinicaltrials.gov/study/NCT02974348
2,CHEMBL112,acetaminophen,Small molecule,Cyclooxygenase inhibitor,Inhibitor,PTGS1,prostaglandin-endoperoxide synthase 1,3.0,Completed,https://clinicaltrials.gov/study/NCT02974348
3,CHEMBL112,acetaminophen,Small molecule,Vanilloid receptor opener,Opener,TRPV1,transient receptor potential cation channel su...,3.0,Completed,https://clinicaltrials.gov/study/NCT02974348
4,CHEMBL628,pentoxifylline,Small molecule,"3',5'-cyclic phosphodiesterase inhibitor",NaN,PDE4A,phosphodiesterase 4A,2.0,Terminated,https://clinicaltrials.gov/study/NCT00133393


## LINCS

### `df_lincs_info`

This dataframe holds information about the signatures (sig_id), and their related perturbagens, cell lines, dose and timepoints.

This data is filtered for specific cells, timepoints and doses to extract the relevant signature IDs `df_lincs_sig` to then filter `df_lincs`

In [ ]:
# Load data
df_lincs_info = pd.read_csv(INPUT + 'GSE92742_Broad_LINCS_sig_info.txt', sep = '\t')

# Rename columns
df_lincs_info.rename(columns = {'pert_id' : 'perturbagen_id', 'pert_iname' : 'perturbagen_name'}, inplace = True)

# Define filters
filter_cells = ['HT29']
filter_timepoints = [6]
filter_doses = [10]

# Filter data
df_lincs_info = df_lincs_info[(df_lincs_info['cell_id'].isin(filter_cells)) &
                              (df_lincs_info['pert_time'].isin(filter_timepoints)) &
                              (df_lincs_info['pert_dose'].isin(filter_doses))]
# Report
num_perturbagens = len(pd.unique(df_lincs_info['perturbagen_name']))

print('>> df_lincs_info')
print(f'Cell line(s): {filter_cells}')
print(f'Timepoint(s): {filter_timepoints}')
print(f'Dose(s): {filter_doses}')
print()
print(f'{num_perturbagens:,} perturbagens found meeting filter criteria')
print()

# Sort data
df_lincs_info.sort_values(by = 'perturbagen_name', inplace = True)
# Save data
#df_lincs_info.to_csv(OUTPUT + 'df_lincs_info.csv', index = False)
# Show data
df_lincs_info.head()

>> df_lincs_info
Cell line(s): ['HT29']
Timepoint(s): [6]
Dose(s): [10]

4,766 perturbagens found meeting filter criteria



C:\Users\roman\AppData\Local\Temp\ipykernel_10124\1354772937.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_lincs_info = pd.read_csv(INPUT + 'GSE92742_Broad_LINCS_sig_info.txt', sep = '\t')


,sig_id,perturbagen_id,perturbagen_name,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
138794,CPC018_HT29_6H:BRD-K06817181-001-01-5:10,BRD-K06817181,"1,2,3,4,5,6-hexabromocyclohexane",trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC018_HT29_6H_X1_B4_DUO52HI53LO:J08|CPC018_HT...
134845,CPC017_HT29_6H:BRD-K74430258-001-01-2:10,BRD-K74430258,"1,2-dichlorobenzene",trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC017_HT29_6H_X1_B4_DUO52HI53LO:D08|CPC017_HT...
98195,CPC010_HT29_6H:BRD-K32795028-001-10-9:10,BRD-K32795028,1-benzylimidazole,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC010_HT29_6H_X1_B3_DUO52HI53LO:E22|CPC010_HT...
138745,CPC018_HT29_6H:BRD-A80928489-001-01-0:10,BRD-A80928489,1-monopalmitin,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC018_HT29_6H_X1_B4_DUO52HI53LO:P22|CPC018_HT...
138887,CPC018_HT29_6H:BRD-K31491153-001-01-2:10,BRD-K31491153,1-phenylbiguanide,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC018_HT29_6H_X1_B4_DUO52HI53LO:L24|CPC018_HT...


#### `df_lincs_perturbagens`

In [10]:
# Load data
df_lincs_info = pd.read_csv(OUTPUT + 'df_lincs_info.csv')

# Isolate columns
df_lincs_perturbagens = df_lincs_info[['perturbagen_id', 'perturbagen_name']].copy()
# Remove duplicates
df_lincs_perturbagens.drop_duplicates(inplace = True, ignore_index = True)

# Save data
df_lincs_perturbagens.to_csv(OUTPUT + 'df_lincs_perturbagens.csv', index = False)
# Show data
df_lincs_perturbagens.head()

,perturbagen_id,perturbagen_name
0,BRD-K06817181,"1,2,3,4,5,6-hexabromocyclohexane"
1,BRD-K74430258,"1,2-dichlorobenzene"
2,BRD-K32795028,1-benzylimidazole
3,BRD-A80928489,1-monopalmitin
4,BRD-K31491153,1-phenylbiguanide


#### `df_lincs_sig`

In [11]:
# Load data
df_lincs_info = pd.read_csv(OUTPUT + 'df_lincs_info.csv')

# Extract signature IDs
df_lincs_sig = pd.DataFrame(df_lincs_info['sig_id'], columns = ['sig_id'])

# Save data
df_lincs_sig.to_csv(OUTPUT + 'df_lincs_sig.csv', index = False)
# Show data
df_lincs_sig.head()

,sig_id
0,CPC018_HT29_6H:BRD-K06817181-001-01-5:10
1,CPC017_HT29_6H:BRD-K74430258-001-01-2:10
2,CPC010_HT29_6H:BRD-K32795028-001-10-9:10
3,CPC018_HT29_6H:BRD-A80928489-001-01-0:10
4,CPC018_HT29_6H:BRD-K31491153-001-01-2:10


### `df_lincs_genes`

This dataframe contains data about all the genes measured, including their ID (rid) and whether they are a landmark gene or not, to be stored in `df_lincs_landmark` and used to filter the resulting graph later on.

In [12]:
# Load data
df_lincs_genes = pd.read_csv(INPUT + 'GSE92742_Broad_LINCS_gene_info.txt', sep = '\t')

# Rename columns
for old, new in zip(['pr_gene_id', 'pr_gene_symbol', 'pr_gene_title', 'pr_is_lm', 'pr_is_bing'], ['rid', 'lincs_name', 'lincs_desc', 'landmark', 'inferred']):
    df_lincs_genes.rename(columns = {old : new}, inplace = True)

# Save data
df_lincs_genes.to_csv(OUTPUT + 'df_lincs_genes.csv', index = False)
# Show data
df_lincs_genes.head()

,rid,lincs_name,lincs_desc,landmark,inferred
0,780,DDR1,discoidin domain receptor tyrosine kinase 1,1,1
1,7849,PAX8,paired box 8,1,1
2,2978,GUCA1A,guanylate cyclase activator 1A,0,0
3,2049,EPHB3,EPH receptor B3,0,1
4,2101,ESRRA,estrogen related receptor alpha,0,1


#### `df_lincs_landmark`

In [13]:
# Load data
df_lincs_landmark = pd.read_csv(OUTPUT + 'df_lincs_genes.csv')

# Filter data
df_lincs_landmark = df_lincs_landmark[df_lincs_landmark['landmark'] == 1]

# Save data
df_lincs_landmark.to_csv(OUTPUT + 'df_lincs_landmark.csv', index = False)
# Show data
df_lincs_landmark.head()

,rid,lincs_name,lincs_desc,landmark,inferred
0,780,DDR1,discoidin domain receptor tyrosine kinase 1,1,1
1,7849,PAX8,paired box 8,1,1
25,6193,RPS5,ribosomal protein S5,1,1
43,23,ABCF1,ATP binding cassette subfamily F member 1,1,1
49,9552,SPAG7,sperm associated antigen 7,1,1


### `df_lincs`

In [14]:
# Load data
df_lincs_sig = pd.read_csv(OUTPUT + 'df_lincs_sig.csv')
# Extract signature IDs
list_lincs_sig = df_lincs_sig['sig_id'].tolist()

# Filter LINCS data
df_lincs = parse.parse(INPUT + 'LDS-1481_1.0.gctx', cid = list_lincs_sig).data_df

# Show data
df_lincs.head()

c:\Users\roman\AppData\Local\Programs\Python\Python310\lib\site-packages\cmapPy\pandasGEXpress\parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))
c:\Users\roman\AppData\Local\Programs\Python\Python310\lib\site-packages\cmapPy\pandasGEXpress\parse_gctx.py:275: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  meta_df = meta_df.apply(lambda x: pd.to_numeric(x, errors="ignore"))


cid,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,CPC004_HT29_6H:BRD-K20482099-001-01-1:10,CPC005_HT29_6H:BRD-K62929068-001-03-3:10,CPC005_HT29_6H:BRD-K43405658-001-01-8:10,CPC004_HT29_6H:BRD-K03670461-001-02-0:10,CPC004_HT29_6H:BRD-K36737713-001-01-6:10,CPC005_HT29_6H:BRD-K51223576-001-01-3:10,CPC004_HT29_6H:BRD-A14966924-001-03-0:10,CPC004_HT29_6H:BRD-K79131256-001-08-8:10,...,PCLB003_HT29_24H:BRD-K64024097-001-02-8:10,PCLB003_HT29_24H:BRD-K64106162-001-02-3:10,PCLB003_HT29_24H:BRD-K16956545-001-01-0:10,PCLB003_HT29_24H:BRD-K78431006-001-05-2:10,PCLB003_HT29_24H:BRD-K63828191-003-23-0:10,PCLB003_HT29_24H:BRD-K93754473-001-17-7:10,PCLB003_HT29_24H:BRD-A52530684-003-01-7:10,PCLB003_HT29_24H:BRD-A75409952-001-01-6:10,PCLB003_HT29_24H:BRD-K04466929-001-05-1:10,PCLB003_HT29_24H:BRD-K17953061-001-10-1:10
rid,,,,,,,,,,,,,,,,,,,,,
5720,-2.999525,-0.623000,0.141888,-0.752267,0.380861,0.076565,-0.749507,-0.284185,0.664252,-0.007322,...,-1.151560,-0.000140,0.589545,-4.135498,-1.515962,-0.059089,2.775635,1.046220,2.3866,0.1313
466,-1.922989,-1.483600,-0.004969,-1.476633,0.828392,0.913175,-0.654432,0.067755,0.684563,1.565995,...,0.390024,0.443335,0.784108,0.183998,-0.215364,1.981584,-1.622547,0.435221,-0.3456,-1.8265
6009,-0.742801,-0.406733,-0.614206,-3.505600,0.381193,0.356511,-0.324873,-0.300248,0.563278,1.232050,...,-0.415272,-0.182099,0.496713,0.417429,2.301637,-0.266856,-2.440649,-0.030544,0.3245,1.5535
2309,-1.567428,0.373000,0.258819,0.750733,-0.013309,-0.172938,-0.571196,0.091755,0.042603,-1.011316,...,0.054832,-0.706762,-0.519995,-0.955052,0.773697,-0.423104,-2.262421,0.163010,0.2551,-2.9476
387,-2.488176,-0.959167,0.737944,-1.150100,-0.292709,-1.357093,0.885794,-0.627101,-0.029998,-0.127312,...,0.611129,0.537418,0.410376,-1.394473,1.186869,-0.120520,-0.581043,-1.971128,-0.7342,-0.3201


#### `df_lincs_meta`

In [15]:
# Extract columns
list_cids = list(df_lincs.columns)

# Initialise dataframe
df_lincs_meta = pd.DataFrame(list_cids, columns = ['cid'])
# Copy column
df_lincs_meta['metadata'] = df_lincs_meta['cid']
# Extract perturbagen ID
df_lincs_meta['metadata'] = df_lincs_meta['metadata'].str.replace(r'-(?:[^-]+-){2}[^-]+(?=:\d+$)', '', regex = True)
# Split metadata column
df_lincs_meta[['data', 'perturbagen_id', 'dose']] = df_lincs_meta['metadata'].str.split(':', expand = True)
# Split data column
df_lincs_meta[['dataset', 'cell_line', 'timepoint']] = df_lincs_meta['data'].str.split('_', expand = True)
# Drop columns
df_lincs_meta.drop(columns = ['metadata', 'data'], inplace = True)

# Load data
df_lincs_perturbagens = pd.read_csv(OUTPUT + 'df_lincs_perturbagens.csv')
# Merge
df_lincs_meta = pd.merge(df_lincs_meta, df_lincs_perturbagens, how = 'left', on = 'perturbagen_id')

# Save data
df_lincs_meta.to_csv(OUTPUT + 'df_lincs_meta.csv', index = False)
# Show data
df_lincs_meta.head()

,cid,perturbagen_id,dose,dataset,cell_line,timepoint,perturbagen_name
0,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,BRD-A85280935,10,CPC005,HT29,6H,quinpirole
1,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,BRD-A07824748,10,CPC005,HT29,6H,flavanone
2,CPC004_HT29_6H:BRD-K20482099-001-01-1:10,BRD-K20482099,10,CPC004,HT29,6H,rutin
3,CPC005_HT29_6H:BRD-K62929068-001-03-3:10,BRD-K62929068,10,CPC005,HT29,6H,6-benzylaminopurine
4,CPC005_HT29_6H:BRD-K43405658-001-01-8:10,BRD-K43405658,10,CPC005,HT29,6H,tyrphostin-AG-527


#### `df_lincs`

In [16]:
# Load data
df_lincs_genes = pd.read_csv(OUTPUT + 'df_lincs_genes.csv')

# Reset index
df_lincs.reset_index(inplace = True)
# Assert datatypes
df_lincs['rid'] = df_lincs['rid'].astype(int)
df_lincs_genes['rid'] = df_lincs_genes['rid'].astype(int)

# Merge
df_lincs = pd.merge(df_lincs, df_lincs_genes[['rid', 'lincs_name', 'lincs_desc', 'landmark']], how = 'left', on = 'rid')
# Melt on gene expression values
df_lincs = df_lincs.melt(id_vars = ['rid', 'lincs_name', 'lincs_desc', 'landmark'], value_vars = list_cids, var_name = 'cid', value_name = 'value')
# Merge with metadata
df_lincs = pd.merge(df_lincs, df_lincs_meta, how = 'left', on = 'cid')
# Reorder columns
df_lincs = df_lincs[['cid', 'dataset', 'cell_line', 'perturbagen_id', 'perturbagen_name', 'dose', 'timepoint', 'rid', 'lincs_name', 'lincs_desc', 'landmark', 'value']]
# Rename columns
df_lincs.rename(columns = {'rid' : 'gene_id', 'value' : 'dexp'}, inplace = True)

# Save data
pickle_save(OUTPUT + 'df_lincs.pkl', df_lincs)
# Show data
df_lincs.head()

,cid,dataset,cell_line,perturbagen_id,perturbagen_name,dose,timepoint,gene_id,lincs_name,lincs_desc,landmark,dexp
0,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,5720,PSME1,proteasome activator subunit 1,1,-2.999525
1,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,466,ATF1,activating transcription factor 1,1,-1.922989
2,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,6009,RHEB,Ras homolog enriched in brain,1,-0.742801
3,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,2309,FOXO3,forkhead box O3,1,-1.567428
4,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,387,RHOA,ras homolog family member A,1,-2.488176


### `df_lincs_dexp`

In [ ]:
# Load data
df_lincs = pickle_load(OUTPUT + 'df_lincs.pkl')

# Get unique perturbagen IDs
len_unique_ids = len(pd.unique(df_lincs['perturbagen_id']))
# Count number of timepoints per perturbagen ID
timepoint_count = df_lincs.groupby(by = 'perturbagen_id')['timepoint'].nunique()
# Count number of perturbagen IDs with 2 timepoint data entries
list_timepoint_ids = timepoint_count[timepoint_count == 2].index.tolist()
# Calculate variables
len_timepoint_ids = len(list_timepoint_ids)
percent_timepoint_ids = len_timepoint_ids / len_unique_ids * 100

# Report
print(f'{percent_timepoint_ids:.2f}% of perturbagens ({len_timepoint_ids:,}/{len_unique_ids:,}) with data at all desired timepoints')

18.36% of perturbagens (938/5,108) with data at all desired timepoints


In [57]:
# Filter data
df_lincs_dexp = df_lincs[df_lincs['perturbagen_id'].isin(list_timepoint_ids)]
df_lincs_dexp = df_lincs_dexp[df_lincs_dexp['landmark'] == 1]

# Save data
# Show data
df_lincs_dexp.head()

,cid,dataset,cell_line,perturbagen_id,perturbagen_name,dose,timepoint,gene_id,lincs_name,lincs_desc,landmark,dexp
0,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,5720,PSME1,proteasome activator subunit 1,1,-2.999525
1,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,466,ATF1,activating transcription factor 1,1,-1.922989
2,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,6009,RHEB,Ras homolog enriched in brain,1,-0.742801
3,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,2309,FOXO3,forkhead box O3,1,-1.567428
4,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005,HT29,BRD-A85280935,quinpirole,10,6H,387,RHOA,ras homolog family member A,1,-2.488176


In [59]:
for pert in list_perturbagens:
    print(pert, len(df_lincs_dexp[df_lincs_dexp['perturbagen_name'] == pert]))

acetaminophen 0
pentoxifylline 0
imatinib 0
rosiglitazone 0
agatolimod 0
agatolimod sodium 0
imiquimod 1956
mefloquine 0


In [58]:
# Get perturbagen list
list_perturbagens = list(pd.unique(df_opentargets['drugName']))
list_perturbagens = list_perturbagens + ['mefloquine']

# # Get n random other perturbagens
# df_slice = df_lincs_info[~df_lincs_info['perturbagen_name'].isin(list_perturbagens)]
# remaining_perturbagens = list(pd.unique(df_slice['perturbagen_name']))
# num_random = 200
# random_perturbagens = random.choices(remaining_perturbagens, k = num_random)

# # Append randoms to perturbagen list
# list_perturbagens = list_perturbagens + random_perturbagens

# # Set filter values
# filter_perturbagens = list_perturbagens